In [1]:
!pip install z3-solver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.0/33.0 MB 44.0 MB/s eta 0:00:00:00:010:01m


In [2]:
from z3 import *

class LogicVerifier:
    def __init__(self):
        # Create a Z3 Solver instance
        self.solver = Solver()
        
    def verify_entailment(self, premises, conclusion):
        """
        Proof by Contradiction: 
        To prove that (Premises => Conclusion) is valid, 
        we add (Premises AND Not(Conclusion)) to the solver.
        If the solver returns 'unsat' (unsatisfiable), the conclusion is mathematically PROVEN.
        If it returns 'sat', the conclusion is INVALID (a counterexample exists).
        """
        self.solver.push() # Save current state
        
        # Add all premises to the solver
        for premise in premises:
            self.solver.add(premise)
            
        # Add the NEGATION of the conclusion
        self.solver.add(Not(conclusion))
        
        # Check satisfiability
        result = self.solver.check()
        
        self.solver.pop() # Restore state
        
        if result == unsat:
            return "✅ PROVEN (Entailed) - The logic is mathematically sound."
        elif result == sat:
            return "❌ INVALID (Not Entailed) - The model hallucinated or made a logical error."
        else:
            return "❓ UNKNOWN - The solver couldn't decide."

print("Logic Verifier Engine initialized successfully!")

Logic Verifier Engine initialized successfully!


In [3]:
# 1. Define the Sort (Type)
Entity = DeclareSort('Entity')

# 2. Define Predicates (Functions that return True/False)
is_boroogove = Function('is_boroogove', Entity, BoolSort())
is_tweequ = Function('is_tweequ', Entity, BoolSort())

# 3. Define Constants (Specific Entities)
caroline = Const('caroline', Entity)
x = Const('x', Entity) # Variable for quantifiers

# 4. Formulate the Premises
premises = [
    # Premise 1: For all x, if x is a boroogove, then x is a tweequ
    ForAll([x], Implies(is_boroogove(x), is_tweequ(x))),
    
    # Premise 2: Caroline is a boroogove
    is_boroogove(caroline)
]

# Initialize our verifier
verifier = LogicVerifier()

print("--- Scenario 1: Checking a SOUND logical conclusion ---")
# Conclusion 1: Caroline is a tweequ
conclusion_1 = is_tweequ(caroline)
result_1 = verifier.verify_entailment(premises, conclusion_1)
print(f"LLM Claim: Caroline is a Tweequ.\nZ3 Result: {result_1}\n")


print("--- Scenario 2: Checking a FALLACY (LLM Hallucination) ---")
# Conclusion 2 (Fallacy): If something is a tweequ, it MUST be Caroline. 
# (LLMs often make this reverse logic mistake).
y = Const('y', Entity)
conclusion_2 = ForAll([y], Implies(is_tweequ(y), y == caroline))
result_2 = verifier.verify_entailment(premises, conclusion_2)
print(f"LLM Claim: Only Caroline is a Tweequ.\nZ3 Result: {result_2}\n")

--- Scenario 1: Checking a SOUND logical conclusion ---
LLM Claim: Caroline is a Tweequ.
Z3 Result: ✅ PROVEN (Entailed) - The logic is mathematically sound.

--- Scenario 2: Checking a FALLACY (LLM Hallucination) ---
LLM Claim: Only Caroline is a Tweequ.
Z3 Result: ❌ INVALID (Not Entailed) - The model hallucinated or made a logical error.

